# CellOracle GRN Inference: Norman 2019 (K562)

**Purpose:** Build K562-specific GRN → extract M matrix for scGPT attention bias (Stage 2).

**Features:**
- Timestamped output dir `celloracle_grn_YYYYMMDD_HHMMSS/`
- Auto-detects and resumes from latest previous run
- Per-stage checkpoints — survives kernel restarts / disconnects
- Log file in output dir

**Design notes:**
- K562 is a single cell type → uses `GRN_unit="whole"` (no cluster-specific GRNs)
- CellOracle is used as a **GRN source**, not a perturbation prediction benchmark
- Embedding is only needed for perturbation simulation (not our use case), but Oracle requires it

## 0. Configuration

In [1]:
import os
os.environ["PATH"] = "/conda_envs/celloracle_env/bin:" + os.environ["PATH"]

import importlib
import pybedtools
import pybedtools.bedtool
importlib.reload(pybedtools.bedtool)
importlib.reload(pybedtools)

# Verify bedtools
a = pybedtools.example_bedtool('a.bed')
b = pybedtools.example_bedtool('b.bed')
print(a.intersect(b))
#this is very important to check if bedtools works

chr1	155	200	feature2	0	+
chr1	155	200	feature3	0	-
chr1	900	901	feature4	0	+



In [2]:
# ============ EDIT THESE PATHS ============
BED_PATH    = "../../data/ATAC_Seq/K562/K562_ATAC_peaks.bed"
NORMAN_H5AD = "../../data/splits/norman_2019_raw_sparse_with_splits_original_from_GEARS.h5ad"
GENOMES_DIR = "../../data/genomes"   # contains hg38/hg38.fa and hg38.fa.fai
REF_GENOME  = "hg38"

# CellOracle params
N_TOP_GENES    = 3000
MAX_CELLS      = 30000
MOTIF_SCORE_TH = 10
MOTIF_FPR      = 0.02
MOTIF_NCPUS    = 8
GRN_ALPHA      = 10
LINK_P_THRESH  = 0.001
LINK_TOP_EDGES = 2000

## 1. Setup: Output Dir, Logging, Resume Detection

In [3]:
import os
import sys
import glob
import logging
from datetime import datetime

# --- Find previous run or create new timestamped dir ---
BASE_PREFIX = "celloracle_grn_"
previous_dirs = sorted(glob.glob(f"{BASE_PREFIX}*"))

RESUME = False
if previous_dirs:
    latest = previous_dirs[-1]
    print(f"Found previous run: {latest}")
    OUTPUT_DIR = latest
    RESUME = True
else:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    OUTPUT_DIR = f"{BASE_PREFIX}{ts}"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"New run: {OUTPUT_DIR}")

print(f"OUTPUT_DIR = {OUTPUT_DIR}")
print(f"RESUME = {RESUME}")

Found previous run: celloracle_grn_norman2019_v3_checkpoint.ipynb
OUTPUT_DIR = celloracle_grn_norman2019_v3_checkpoint.ipynb
RESUME = True


In [4]:
# --- Find previous run or create new timestamped dir ---
BASE_PREFIX = "celloracle_grn_"
previous_dirs = sorted([d for d in glob.glob(f"{BASE_PREFIX}*/") if os.path.isdir(d)])

RESUME = False
if previous_dirs:
    latest = previous_dirs[-1].rstrip("/")
    print(f"Found previous run: {latest}")
    OUTPUT_DIR = latest
    RESUME = True
else:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    OUTPUT_DIR = f"{BASE_PREFIX}{ts}"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"New run: {OUTPUT_DIR}")

Found previous run: celloracle_grn_20260305_215044


In [5]:
# --- Checkpoint helper ---
def ckpt_path(name):
    """Return full path for a checkpoint file."""
    return os.path.join(OUTPUT_DIR, name)

def ckpt_exists(name):
    """Check if checkpoint file exists."""
    p = ckpt_path(name)
    exists = os.path.exists(p)
    if exists:
        logger.info(f"Checkpoint found: {name}")
    return exists

In [6]:
# --- Verify inputs ---
for p, label in [
    (BED_PATH, "BED"),
    (NORMAN_H5AD, "Norman h5ad"),
    (f"{GENOMES_DIR}/{REF_GENOME}/{REF_GENOME}.fa", "Genome FASTA"),
]:
    assert os.path.exists(p), f"Missing {label}: {p}"
    logger.info(f"Input OK: {label} -> {p}")

NameError: name 'logger' is not defined

In [ ]:
# --- Imports ---
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import matplotlib.pyplot as plt
import celloracle as co
from celloracle import motif_analysis as ma

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
plt.rcParams['figure.figsize'] = [6, 4.5]
plt.rcParams['savefig.dpi'] = 300

logger.info(f"CellOracle version: {co.__version__}")

## 2. Bulk ATAC-seq → TSS Annotation

In [ ]:
CKPT_TSS = "K562_processed_peak_file.csv"

if ckpt_exists(CKPT_TSS):
    tss_annotated_df = pd.read_csv(ckpt_path(CKPT_TSS))
    logger.info(f"Loaded TSS checkpoint: {tss_annotated_df.shape[0]} peak-gene pairs")
else:
    logger.info("Stage 2: BED -> TSS annotation")
    bed = ma.read_bed(BED_PATH)
    logger.info(f"Peaks loaded: {bed.shape[0]}")

    peaks = ma.process_bed_file.df_to_list_peakstr(bed)
    logger.info(f"Total peaks: {len(peaks)}")

    # Ensure genome symlinks exist
    default_genome_dir = os.path.expanduser(f"~/.local/share/genomes/{REF_GENOME}")
    os.makedirs(default_genome_dir, exist_ok=True)
    src_fa  = os.path.abspath(f"{GENOMES_DIR}/{REF_GENOME}/{REF_GENOME}.fa")
    src_fai = os.path.abspath(f"{GENOMES_DIR}/{REF_GENOME}/{REF_GENOME}.fa.fai")
    for src in [src_fa, src_fai]:
        dst = os.path.join(default_genome_dir, os.path.basename(src))
        if not os.path.exists(dst):
            os.symlink(src, dst)
            logger.info(f"Symlinked: {dst}")
    sizes_file = os.path.join(default_genome_dir, f"{REF_GENOME}.fa.sizes")
    if not os.path.exists(sizes_file):
        fai = pd.read_csv(src_fai, sep='\t', header=None, usecols=[0,1], names=['chrom','size'])
        fai.to_csv(sizes_file, sep='\t', header=False, index=False)

    tss_annotated = ma.get_tss_info(peak_str_list=peaks, ref_genome=REF_GENOME)
    logger.info(f"TSS-annotated peaks: {tss_annotated.shape[0]}")

    peak_id_tss = ma.process_bed_file.df_to_list_peakstr(tss_annotated)
    tss_annotated_df = pd.DataFrame({
        "peak_id": peak_id_tss,
        "gene_short_name": tss_annotated.gene_short_name.values
    }).reset_index(drop=True)

    tss_annotated_df.to_csv(ckpt_path(CKPT_TSS), index=False)
    logger.info(f"Saved TSS checkpoint: {tss_annotated_df.shape[0]} peak-gene pairs, {tss_annotated_df['gene_short_name'].nunique()} genes")

## 3. TF Motif Scan → Base GRN

In [ ]:
CKPT_TFINFO = "K562_TFinfo.celloracle.tfinfo"
CKPT_BASE_GRN = "K562_base_GRN.parquet"

if ckpt_exists(CKPT_BASE_GRN):
    base_GRN_df = pd.read_parquet(ckpt_path(CKPT_BASE_GRN))
    logger.info(f"Loaded base GRN: {base_GRN_df.shape[0]} peaks x {base_GRN_df.shape[1]-2} TFs")
else:
    # Check for motif scan checkpoint
    if ckpt_exists(CKPT_TFINFO):
        logger.info("Loading motif scan checkpoint")
        tfi = ma.load_TFinfo_from_parquets(ckpt_path(CKPT_TFINFO))
    else:
        logger.info("Stage 3: Motif scan (this may take 30-60 min)")
        tfi = ma.TFinfo(
            peak_data_frame=tss_annotated_df,
            ref_genome=REF_GENOME,
            genomes_dir=GENOMES_DIR
        )
        tfi.scan(fpr=MOTIF_FPR, motifs=None, verbose=True, n_cpus=MOTIF_NCPUS)
        tfi.save_as_parquet(ckpt_path(CKPT_TFINFO))
        logger.info(f"Motif scan checkpoint saved")

    # Filter and build base GRN
    tfi.reset_filtering()
    tfi.filter_motifs_by_score(threshold=MOTIF_SCORE_TH)
    tfi.make_TFinfo_dataframe_and_dictionary(verbose=True)

    base_GRN_df = tfi.to_dataframe()
    base_GRN_df.to_parquet(ckpt_path(CKPT_BASE_GRN))
    logger.info(f"Base GRN saved: {base_GRN_df.shape[0]} peaks x {base_GRN_df.shape[1]-2} TFs")

## 4. Prepare Norman 2019 scRNA-seq

K562 is one cell type. We use ctrl cells only, preprocess, assign single cluster.

In [ ]:
CKPT_ADATA = "K562_ctrl_preprocessed.h5ad"

if ckpt_exists(CKPT_ADATA):
    adata_co = sc.read_h5ad(ckpt_path(CKPT_ADATA))
    logger.info(f"Loaded preprocessed adata: {adata_co.shape}")
else:
    logger.info("Stage 4: Preprocessing Norman 2019 ctrl cells")
    adata = sc.read_h5ad(NORMAN_H5AD)
    logger.info(f"Full data: {adata.shape}, conditions: {adata.obs['condition'].nunique()}")

    # Subset to pure ctrl cells only
    adata_ctrl = adata[adata.obs['condition'] == 'ctrl'].copy()
    logger.info(f"Pure ctrl cells: {adata_ctrl.shape[0]}")

    # Fallback: include gene+ctrl singles if too few pure ctrl
    if adata_ctrl.shape[0] < 1000:
        logger.warning("Few pure ctrl cells, including gene+ctrl singles")
        ctrl_labels = [c for c in adata.obs['condition'].unique()
                       if c == 'ctrl' or ('+ctrl' in c or 'ctrl+' in c)]
        adata_ctrl = adata[adata.obs['condition'].isin(ctrl_labels)].copy()
        logger.info(f"Expanded ctrl cells: {adata_ctrl.shape[0]}")

    # Verify raw counts
    xmax = adata_ctrl.X.max()
    logger.info(f"X max={xmax:.1f}, dtype={adata_ctrl.X.dtype}")

    # Preprocess
    adata_co = adata_ctrl.copy()
    adata_co.layers['raw_count'] = adata_co.X.copy()

    sc.pp.normalize_total(adata_co, target_sum=1e4)
    sc.pp.log1p(adata_co)
    sc.pp.highly_variable_genes(adata_co, n_top_genes=N_TOP_GENES)
    adata_co = adata_co[:, adata_co.var['highly_variable']].copy()
    logger.info(f"After HVG: {adata_co.shape}")

    # PCA + neighbors
    sc.pp.scale(adata_co, max_value=10)
    sc.tl.pca(adata_co, svd_solver='arpack', n_comps=50)
    sc.pp.neighbors(adata_co, n_neighbors=15, n_pcs=30)

    # Single cluster for K562 (no trajectory for cell line)
    adata_co.obs['cluster'] = 'K562'
    adata_co.obs['cluster'] = adata_co.obs['cluster'].astype('category')
    adata_co.uns['cluster_colors'] = ['#1f77b4']
    logger.info("Assigned single cluster 'K562'")

    # Downsample if needed
    if adata_co.shape[0] > MAX_CELLS:
        sc.pp.subsample(adata_co, n_obs=MAX_CELLS, random_state=42)
        logger.info(f"Downsampled to {adata_co.shape[0]}")

    # UMAP as embedding (CellOracle requires one; only used for simulation viz, not GRN)
    sc.tl.umap(adata_co)
    logger.info(f"UMAP computed. obsm keys: {list(adata_co.obsm.keys())}")

    adata_co.write_h5ad(ckpt_path(CKPT_ADATA))
    logger.info(f"Saved preprocessed adata: {adata_co.shape}")

## 5. CellOracle GRN Inference

In [ ]:
CKPT_ORACLE = "K562_norman.celloracle.oracle"

if ckpt_exists(CKPT_ORACLE):
    oracle = co.load_hdf5(ckpt_path(CKPT_ORACLE))
    logger.info("Loaded Oracle checkpoint")
else:
    logger.info("Stage 5: Building Oracle object")

    # --- Gene name overlap diagnostic ---
    grn_genes = set(base_GRN_df['gene_short_name'])
    adata_genes = set(adata_co.var_names)
    overlap = grn_genes & adata_genes
    logger.info(f"Gene overlap: base_GRN={len(grn_genes)}, adata={len(adata_genes)}, overlap={len(overlap)}")

    if len(overlap) == 0:
        logger.warning("Zero overlap! Scanning adata.var columns for gene symbols...")
        for col in adata_co.var.columns:
            test_overlap = len(grn_genes & set(adata_co.var[col].astype(str)))
            if test_overlap > 0:
                logger.info(f"  var['{col}']: {test_overlap} matches")
        # Auto-remap if a good match found
        for col in ['gene_name', 'gene_short_name', 'symbol', 'gene_symbols']:
            if col in adata_co.var.columns:
                test = len(grn_genes & set(adata_co.var[col].astype(str)))
                if test > 100:
                    logger.info(f"Remapping var_names to var['{col}'] ({test} matches)")
                    adata_co.var_names = adata_co.var[col].astype(str).values
                    adata_co.var_names_make_unique()
                    break

    # --- Create Oracle ---
    oracle = co.Oracle()
    adata_co.X = adata_co.layers['raw_count'].copy()

    # Pick embedding (only for viz, not GRN)
    embedding_name = "X_umap"
    if embedding_name not in adata_co.obsm:
        embedding_name = list(adata_co.obsm.keys())[0]
        logger.warning(f"X_umap not found, using {embedding_name}")

    oracle.import_anndata_as_raw_count(
        adata=adata_co,
        cluster_column_name="cluster",
        embedding_name=embedding_name
    )
    logger.info("Oracle: anndata imported")

    oracle.import_TF_data(TF_info_matrix=base_GRN_df)
    logger.info("Oracle: base GRN imported")

    # Verify overlap
    n_reg = oracle.adata.var.get('isin_TFdict_regulators', pd.Series(dtype=bool)).sum()
    n_tgt = oracle.adata.var.get('isin_TFdict_targets', pd.Series(dtype=bool)).sum()
    logger.info(f"Oracle overlap: regulators={n_reg}, targets={n_tgt}")
    if n_reg == 0 or n_tgt == 0:
        logger.error("FATAL: No gene overlap. Fix gene names before proceeding.")
        raise ValueError("No overlap between base GRN and adata var_names.")

    # PCA for KNN
    oracle.perform_PCA()
    n_comps = min(
        np.where(np.diff(np.diff(np.cumsum(oracle.pca.explained_variance_ratio_)) > 0.002))[0][0],
        50
    )
    logger.info(f"PCA: {n_comps} components")

    # KNN imputation
    n_cell = oracle.adata.shape[0]
    k = max(int(0.025 * n_cell), 30)
    logger.info(f"KNN imputation: n_cell={n_cell}, k={k}")
    oracle.knn_imputation(
        n_pca_dims=n_comps, k=k, balanced=True,
        b_sight=k*8, b_maxl=k*4, n_jobs=4
    )
    logger.info("KNN imputation done")

    oracle.to_hdf5(ckpt_path(CKPT_ORACLE))
    logger.info("Oracle checkpoint saved")

In [ ]:
CKPT_LINKS = "K562_norman.celloracle.links"

if ckpt_exists(CKPT_LINKS):
    links = co.load_hdf5(ckpt_path(CKPT_LINKS))
    logger.info("Loaded Links checkpoint")
else:
    logger.info("Stage 5b: GRN inference (GRN_unit='whole', single cell type)")
    links = oracle.get_links(
        cluster_name_for_GRN_unit="whole",
        alpha=GRN_ALPHA,
        verbose_level=10
    )
    logger.info("GRN inference done")

    links.filter_links(p=LINK_P_THRESH, weight="coef_abs", threshold_number=LINK_TOP_EDGES)
    links.to_hdf5(ckpt_path(CKPT_LINKS))
    logger.info("Links checkpoint saved")

## 6. Extract M Matrix

In [ ]:
CKPT_M = "K562_GRN_M_matrix.csv"

if ckpt_exists(CKPT_M):
    M_matrix = pd.read_csv(ckpt_path(CKPT_M), index_col=0)
    logger.info(f"Loaded M matrix: {M_matrix.shape}")
else:
    logger.info("Stage 6: Extracting M matrix")

    all_edges = []
    for cname, grn in links.links_dict.items():
        g = grn.copy()
        g['cluster'] = cname
        all_edges.append(g)
    all_edges_df = pd.concat(all_edges, ignore_index=True)
    logger.info(f"Total edges: {all_edges_df.shape[0]}")

    mean_edges = all_edges_df.groupby(['source','target']).agg(
        coef_mean=('coef_mean','mean'),
        coef_abs_mean=('coef_abs','mean'),
        p_min=('p','min')
    ).reset_index()
    logger.info(f"Unique edges: {mean_edges.shape[0]}, TFs: {mean_edges['source'].nunique()}, Targets: {mean_edges['target'].nunique()}")

    sig_edges = mean_edges[mean_edges['p_min'] < LINK_P_THRESH].copy()
    logger.info(f"Significant edges (p<{LINK_P_THRESH}): {sig_edges.shape[0]}")

    M_matrix = sig_edges.pivot_table(
        index='target', columns='source',
        values='coef_mean', fill_value=0.0
    )
    logger.info(f"M shape: {M_matrix.shape}, non-zero: {(M_matrix!=0).sum().sum()}, sparsity: {1-(M_matrix!=0).sum().sum()/M_matrix.size:.4f}")

    # Save all outputs
    M_matrix.to_csv(ckpt_path(CKPT_M))
    np.save(ckpt_path("K562_GRN_M_matrix.npy"), M_matrix.values)
    pd.Series(M_matrix.index, name='target_gene').to_csv(
        ckpt_path("K562_GRN_M_target_genes.csv"), index=False)
    pd.Series(M_matrix.columns, name='source_TF').to_csv(
        ckpt_path("K562_GRN_M_source_TFs.csv"), index=False)
    mean_edges.to_csv(ckpt_path("K562_GRN_mean_edges.csv"), index=False)
    logger.info("All M matrix files saved")

## 7. Sanity Checks

In [ ]:
me = pd.read_csv(ckpt_path("K562_GRN_mean_edges.csv"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(me['coef_mean'], bins=100)
axes[0].set_title('Mean GRN coefficients')
axes[1].hist(np.log10(me['p_min']+1e-300), bins=100)
axes[1].axvline(np.log10(LINK_P_THRESH), c='r', ls='--', label=f'p={LINK_P_THRESH}')
axes[1].set_title('log10(min p-value)')
axes[1].legend()
plt.tight_layout()
plt.savefig(ckpt_path("sanity_coef_pval.png"), dpi=150, bbox_inches='tight')
plt.show()
logger.info("Saved sanity_coef_pval.png")

In [ ]:
# Top TFs by out-degree
sig = me[me['p_min'] < LINK_P_THRESH]
tf_degree = sig.groupby('source')['target'].nunique().sort_values(ascending=False)
logger.info(f"Top 20 TFs by out-degree:\n{tf_degree.head(20).to_string()}")
print("Top 20 TFs (# targets):")
print(tf_degree.head(20))

In [ ]:
# Overlap with Norman perturbed genes
adata_full = sc.read_h5ad(NORMAN_H5AD)
perturbed_genes = set()
for c in adata_full.obs['condition'].unique():
    if c != 'ctrl':
        for g in c.split('+'):
            if g.strip() != 'ctrl':
                perturbed_genes.add(g.strip())

grn_tfs = set(M_matrix.columns)
overlap = perturbed_genes & grn_tfs
missing = perturbed_genes - grn_tfs
logger.info(f"Perturbed: {len(perturbed_genes)}, GRN TFs: {len(grn_tfs)}, Overlap: {len(overlap)} ({len(overlap)/len(perturbed_genes)*100:.1f}%)")
if missing:
    logger.info(f"Missing from GRN ({len(missing)}): {missing}")
    print(f"Missing ({len(missing)}): {missing}")

In [ ]:
# Final output listing
logger.info("=== Final outputs ===")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
    logger.info(f"  {f} ({size_mb:.1f} MB)")

logger.info("=== Pipeline complete ===")

## Output Summary

| File | Purpose |
|---|---|
| `K562_GRN_M_matrix.npy` | **M for Stage 2**: `attn_scores += alpha * M_grn` |
| `K562_GRN_M_matrix.csv` | Same, readable |
| `K562_GRN_M_target_genes.csv` | Row index |
| `K562_GRN_M_source_TFs.csv` | Column index |
| `K562_GRN_mean_edges.csv` | Full edge list |
| `K562_base_GRN.parquet` | Base GRN |
| `K562_TFinfo.celloracle.tfinfo` | Motif scan checkpoint |
| `K562_norman.celloracle.oracle` | Oracle checkpoint |
| `K562_norman.celloracle.links` | GRN links |
| `celloracle_grn.log` | Full run log |